# SpikeInterface Spike Sorting and Post-Processing Pipeline 
---
**Original:** https://github.com/SpikeInterface/spiketutorials/tree/master/Official_Tutorial_SI_0.99_Nov23

**Author:** Juliette Leclerc (Ph.D. Candidate, Université de Rouen Normandie)

## Overview
This notebook provides a comprehensive, production-ready pipeline for *in vivo* electrophysiology data processing, from raw signals to metrics export and neural trajectories.

## Acknowledgments & Custom Contributions
The core architecture of this pipeline builds upon the foundational tutorials provided by the SpikeInterface team (v0.99). It has been significantly extended and customized to meet our exprimental design.

## **Key custom features and improvements include:**
* **VNS Artifact Management:** Implementation of chunking/blanking strategies to suppress Vagus Nerve Stimulation artifacts.
* **Anatomical Mapping:** Integration with `ProbeInterface` to map signals onto our silicon probe geometries.
* **State-of-the-Art Sorting and Curation:** Optimized pipeline for **Kilosort 4 (KS4)** and automated export to SpikeInterface GUI and Phy.
* **Metrics Export & Population Dynamics**: Automated extraction of curated quality metrics alongside advanced temporal visualizations such as PCA state space to track network-level responses to VNS.

# Table of contents
* [1. Preparation](#preparation)
* [2. Loading recording and defining parameters](#loading)
* [3. Preprocessing](#preprocessing)
* [4. Saving and loading SpikeInterface objects](#save-load)
* [5. Spike sorting](#spike-sorting)
* [6. Extracting spiking unit metrics and visual inspection](#spiking-unit-metrics)
* [7. Curation](#curation)
* [8. Data inspection and classification INT/PYR](#INT_PYR_classification)

# 1. Preparation <a class="anchor" id="preparation"></a>

If you run into troubles at the following step, put this line in CMD : 
"!conda install -c conda-forge ipympl"

In [ ]:
# 1. Standard library imports
import os
import warnings
from pathlib import Path
import json

# 2. Third-party data science imports
import numpy as np
import pandas as pd
import scipy
import sympy as sp
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from mpl_toolkits.mplot3d import Axes3D
import ipywidgets as widgets
from IPython.display import display
import shutil
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from scipy.ndimage import gaussian_filter1d
import tkinter as tk
from tkinter import filedialog

# 3. SpikeInterface & ProbeInterface imports
import spikeinterface.full as si
import spikeinterface.widgets as wi
import probeinterface as pi
from probeinterface.plotting import plot_probe
from spikeinterface_gui import run_mainwindow

# 4. Configuration
warnings.simplefilter("ignore")
print(f"SpikeInterface version: {si.__version__}")

# 2. Loading recording and defining parameters <a class="anchor" id="loading"></a>
More documention on this section here: https://spikeinterface.readthedocs.io/en/latest/modules/core.html

## 2.1 Define data path

First, we have to define our path folder (aka where are the open Ephys folders or the `continuous.dat` file

In [ ]:
# Define experiment identifiers
mice_name = 'H6b-VNS1-J7-6F_2025-02-14'         # e.g., 'H6b-sVNS-J7-6M_2025-03-11'
recording_name = '2025-02-27_18-38-01'    # e.g., '2025-03-11_18-36-34'
OE_node = 'Record Node 101'      # OpenEphys specific recording node


In [ ]:
## Interactive Folder Selection with Tkinter
# Create a hidden main window
root = tk.Tk()
root.withdraw() 
# Force the pop-up to appear on top of other windows (like your browser)
root.attributes('-topmost', True)

print("📂 Please select the 'Data' root folder from the dialog window...")
selected_dir = filedialog.askdirectory(title="Select the Data Root Folder")

# Check if the user actually selected a folder or clicked "Cancel"
if selected_dir:
    Data_root = Path(selected_dir)
    print(f"✅ Data root set to: {Data_root}")
else:
    # Fallback to the relative path if the user cancels
    print("⚠️ No folder selected. Falling back to default './data'")
    Data_root = Path('./data')

## Construct and check paths
base_folder = Data_root / mice_name
oe_folder = base_folder / recording_name / OE_node

# Safety check: Warn the user if the folder is missing
if not oe_folder.exists():
    print(f"⚠️ Warning: The path '{oe_folder}' does not exist.")
    print(f"Please ensure your data is organized as: {Data_root} / {mice_name} / {recording_name} / {OE_node}")
else:
    print(f"✅ Successfully located data folder: {oe_folder}")

## 2.2 Define file parameters

### Define cortex to analyze

In [ ]:
cortex = 'peri' #"peri' or 'contra'
cortex_folder = base_folder / cortex
print(cortex_folder)

### Define bad channels

In [ ]:
bad_channels_num = [25] 
bad_channels = [f"CH{num}" for num in bad_channels_num]

 ### Define VNS_Start_index and start_time

In [ ]:
start_time = 0 #50*60*20000
VNS_start_index =  int(1500612) #int(1500000 + 0*1250) #Found on Matlab for oscillation analysis. Should be based on 1250Hz sampling rate

## 2.3 Preparing Recordings

The `read_openephys()` or 'read_binary' function returns a `Recording` (or `RecordingExtractor`) object. We can print it to visualize some of its properties:

In [ ]:
## If it is the whole Open Ephys folder
full_raw_rec = si.read_openephys(oe_folder)

The `get_traces()` function returns a TxN numpy array where N is the number of channel ids passed in (all channel ids are passed in by default) and T is the number of frames (determined by start_frame and end_frame).

In [ ]:
fs = full_raw_rec.get_sampling_frequency()
trace_snippet = full_raw_rec.get_traces(start_frame=int(fs*0), end_frame=int(fs*2))

### Define the cortex hemisphere in which the signal was recorded.

In [ ]:
#Defining headstage1 = peri and headstage2 = contra
full_raw_rec.set_property(key="cortex", values=["peri"] * 32 + ["contra"] * 32)
quality_values = full_raw_rec.get_property("cortex")
print('Quality_values:', quality_values)

### Slice data according to different data time

In [ ]:
# Define VNS_start, start_time, end_time, in seconds
VNS_start = VNS_start_index*fs/1250 #look at VNS_start_index
VNS_timelength = 30*60*fs
pre_post_length = 15*60*fs #15 min

full_raw_rec.annotate(VNS_start = VNS_start)

pre_full_raw_rec = full_raw_rec.frame_slice(start_time,start_time+pre_post_length) #time_slice(start_time: float | None, end_time: float), in seconds
VNS_full_raw_rec = full_raw_rec.frame_slice(start_time+VNS_start , start_time+VNS_start+VNS_timelength) 
post_full_raw_rec= full_raw_rec.frame_slice(start_time+VNS_start+VNS_timelength , start_time+VNS_start+VNS_timelength+pre_post_length) 

### Give each sliced recordings their timing property

In [ ]:
pre_full_raw_rec.set_property(key="timing", values=["pre"] * 64)
VNS_full_raw_rec.set_property(key="timing", values=["VNS"] * 64)
post_full_raw_rec.set_property(key="timing", values=["post-VNS"] * 64)

full_raw_rec2 = si.append_recordings([pre_full_raw_rec,VNS_full_raw_rec,post_full_raw_rec])
print(full_raw_rec2)

recordings = full_raw_rec2.split_by(property="cortex")
print(recordings)
print(recordings['peri'].get_channel_ids())
print(recordings['contra'].get_channel_ids())

## 2.4 Importing our probe with ProbeInterface
Before moving on with the analysis, we have to load the probe information. For this we will use the [ProbeInterface](https://probeinterface.readthedocs.io/en/main/index.html) package. 

In [ ]:
manufacturer = 'cambridgeneurotech'
probe_name = 'ASSY-37-H6b'

probe = pi.get_probe(manufacturer, probe_name)
print(probe)

In [ ]:
# Manually adding our probe indexes designs.
channel_indices = np.array([29, 19, 18, 28, 30, 20, 17, 21, 31, 22, 16, 23, 27, 26, 25, 24, 7, 6, 5, 4, 8, 10, 9, 3, 11, 2, 12, 1, 13, 0, 14, 15])
probe.set_device_channel_indices(channel_indices)

# Associate recording to probe
raw_rec = recordings[cortex].set_probe(probe)

The probe now has contact ids `id#` and device ids `dev#`. When loading the probe, the device indices (and all the other contact properties) are automatically sorted. And very importantly the recording is reduce to 32 channels.

In [ ]:
#Plotting our probe design with device and contact indexes
fig, ax = plt.subplots(figsize=(14, 10))
plot_probe(probe, ax=ax, with_contact_id=True, with_device_index=True,)
ax.set_xlim(-100, 100)
ax.set_ylim(-50, 400)

# To visualize the channel_id (channel name) from open ephys.
probe_rec = raw_rec.get_probe()
probe_rec.to_dataframe(complete=True).loc[:, ["contact_ids", "device_channel_indices"]]

fig, ax = plt.subplots(figsize=(14, 10))
si.plot_probe_map(raw_rec, with_channel_ids=True, ax=ax)
ax.set_xlim(-100, 100)
ax.set_ylim(-50, 400)
plt.show()

The `widgets` module includes several convenient plotting functions that can be used to explore the data

In [ ]:
# To plot traces before filtering
%matplotlib widget
si.plot_traces(raw_rec, segment_index=0, backend="ipywidgets", mode='line', channel_ids=raw_rec.channel_ids[::8])
plt.show()

# 3. Preprocessing <a class="anchor" id="preprocessing"></a>

Now that the probe information is loaded we can do some preprocessing using `preprocessing` module.

We can filter the recordings, rereference the signals to remove noise, discard noisy channels, whiten the data, remove stimulation artifacts, etc. 

For this notebook, we will filter the recordings and apply common median reference (CMR). All preprocessing modules return new `Recording` objects that apply the underlying preprocessing function. This allows users to access the preprocessed data in the same way as the raw data.

More documention on this section here: https://spikeinterface.readthedocs.io/en/latest/modules/preprocessing.html


In [ ]:
#Bandpass filter the recording and apply common median reference to the original recording
recording_f = si.bandpass_filter(raw_rec, freq_min=300, freq_max=9000)

## 3.1 Removing bad channels and Common Median Reference (CMR):



In [ ]:
# Verifying if bad channel have been indicated above
if bad_channels == []:
    print("No bad channels indicated")
    recording_good_channels = si.common_reference(recording_f, reference='global', operator='median')
else:
    recording_good_channels_f = recording_f.remove_channels(bad_channels)
    recording_good_channels = si.common_reference(recording_good_channels_f, reference='global', operator='median')
    print(recording_good_channels)
    print(recording_good_channels.channel_ids)

#Reset timing for interference removal to consider t=0 at 0
recording_good_channels.reset_times()

**NOTE**: you can also set a default plotting backend with the `si.set_default_plotter_backend()` function.

## 3.2 Isolating VNS interferences
VNS creates small sharp rhytmic interferences that can be easily isolated without impacting spike detection around them.

In [ ]:
#Dawson
VNS_artifact_start = 0.005  # in seconds
artifact_duration = 0.02  # in seconds
time_btw_burst_starts = 0.0358  # in seconds
time_btw_train_starts = 10  # in seconds
n_VNS_trains = 180
n_bursts_per_train = 14
VNS_drift = +0.00045  # in seconds

In [ ]:
# Get all the needed values 
samples_VNS = recording_good_channels.get_num_samples(segment_index= 1) # Number of points in VNS recording

# Sampling frequency
fs = recording_good_channels.get_sampling_frequency()  # Get the sampling rate

# Calculate total treatment time in samples
VNS_treatment_time = recording_good_channels.get_num_samples(segment_index=1)

# Initialize timepoint array
timepoint_array = []

# Create timepoint array
for train in range(n_VNS_trains):
    for burst in range(n_bursts_per_train):
        # Calculate the new timepoint
        new_timepoint = (
            VNS_artifact_start
            + time_btw_burst_starts * burst
            + time_btw_train_starts * train
            + VNS_drift * train
        )
        # Convert time to indices
        if new_timepoint < 0 :
            start_idx = 0
        else :
            start_idx = int(new_timepoint * fs)
            
        end_idx = int((new_timepoint + artifact_duration) * fs)
        timepoint_array.append([start_idx, end_idx])

# Convert to numpy array
timepoint_array = np.array(timepoint_array)

# Save the timepoint array for later
timepoint_array_filename = cortex_folder / f"timepoint_array.npy"
np.save(timepoint_array_filename, timepoint_array)

# Initialize the timepoint_plot_array
timepoint_plot_array = np.zeros(VNS_treatment_time, dtype=int)

# Fill binary array
for start_idx, end_idx in timepoint_array:
    timepoint_plot_array[start_idx:end_idx] = 10000


In [ ]:
# Time range for plotting (in seconds)
time_range = [29*60+50, 29*60+51]
#time_range = [19*60+50, 22*60+50.2]
#time_range = [0, 1]
%matplotlib inline

# Plot the LFP traces
widget = si.plot_traces(
    recording_good_channels,
    segment_index=1,
    backend="matplotlib",
    time_range=time_range,
    mode='line',
    channel_ids=raw_rec.channel_ids[:1]
)

# Access the matplotlib figure and axes
fig = widget.figure
ax = widget.ax

# Sampling frequency
fs = recording_good_channels.get_sampling_frequency()  # Sampling rate in Hz

# Convert time_range to sample indices
start_index = int(time_range[0] * fs)
end_index = int(time_range[1] * fs)

# Generate the time array corresponding to binaryArray[start_index:end_index]
num_samples = end_index - start_index  # Total samples in the range
time_array = np.linspace(time_range[0], time_range[1], num_samples)

# Plot your array on top of the LFP traces
ax.plot(time_array, timepoint_plot_array[start_index:end_index], color='green', linestyle='-', linewidth=1.5, label='Cut sections')

# Add a legend for the overlay
ax.legend()

# Show the plot
plt.show()


In [ ]:
# Select the second segment of the recording
VNS_recording_good_channels = si.select_segment_recording(recording_good_channels, 1)

# Initialize a list to collect isolated segments
isolated_segments = []

# Loop through timepoint_array to slice and collect segments
for inter_VNS_segment in range(len(timepoint_array)):
    if inter_VNS_segment == 0:
        # First segment: slice from the end of the first burst to the start of the second
        new_isolated_segment = VNS_recording_good_channels.frame_slice(
            timepoint_array[inter_VNS_segment, 1],
            timepoint_array[inter_VNS_segment + 1, 0]
        )
    elif inter_VNS_segment == len(timepoint_array) - 1:
        # Last segment: slice from the end of the last burst to the end of the recording
        new_isolated_segment = VNS_recording_good_channels.frame_slice(
            timepoint_array[inter_VNS_segment, 1],
            samples_VNS
        )
    else:
        # Middle segments: slice between consecutive bursts
        new_isolated_segment = VNS_recording_good_channels.frame_slice(
            timepoint_array[inter_VNS_segment, 1],
            timepoint_array[inter_VNS_segment + 1, 0]
        )
    
    # Collect the isolated segment
    isolated_segments.append(new_isolated_segment)

# Concatenate all collected segments in one step
sliced_recording_good_channels = si.ConcatenateSegmentRecording(isolated_segments)


In [ ]:
%matplotlib inline
w = si.plot_traces(sliced_recording_good_channels, segment_index=0, time_range = [0,30], backend="matplotlib")
plt.show()

In [ ]:
sliced_recording_good_channels.get_time_info

In [ ]:
# Reassemble segments back together
recording_final_Pre = si.select_segment_recording(recording_good_channels, 0)
recording_final_Post = si.select_segment_recording(recording_good_channels, 2)
recording_final_VNS = si.select_segment_recording(sliced_recording_good_channels, 0)
recording_final = si.append_recordings([recording_final_Pre,recording_final_VNS,recording_final_Post])

recording_final.annotate(VNS_artifact_start = VNS_artifact_start)
recording_final.annotate(VNS_drift = VNS_drift)
recording_final.annotate(artifact_duration = artifact_duration)

recording_final.get_time_info

# 4. Saving and loading SpikeInterface objects <a class="anchor" id="save-load"></a>

All operations in SpikeInterface are *lazy*, meaning that they are not performed if not needed. This is why the creation of our filter recording was almost instantaneous. However, to speed up further processing, we might want to **save** it to a file and perform those operations (eg. filters, CMR, etc.) at once. 

In [ ]:
n_cpus = os.cpu_count()
n_jobs = n_cpus - 4
print(n_jobs)

Basically, this is all about parrallel processing, and how the data are cut to be processed faster by the sorter, nothing to be changed here but here are some details : 
**n_jobs** = Number of jobs to use. With -1 the number of jobs is the same as number of cores
**chunk_duration** = Number of samples per chunk

In [ ]:
job_kwargs = dict(n_jobs=n_jobs, chunk_duration="1s", progress_bar=True)

**Note**: you can use the `si.set_global_job_kwargs()` to set `job_kwargs` globally for the entire session!

In [ ]:
# Reset timing of recordings
got_time = recording_good_channels.get_times(segment_index=1)
new_VNS_time = got_time[0:recording_final.get_num_samples(segment_index= 1)]

recording_final.set_times(times=new_VNS_time,segment_index=1)
recording_final.reset_times()

# 5. Spike sorting <a class="anchor" id="spike-sorting"></a>

We can now run spike sorting on the above recording. 

We will sort the bandpass cached filtered recording the `recording_to_analyze` object.

More documention on this section here: https://spikeinterface.readthedocs.io/en/latest/modules/sorters.html

In [ ]:
# Verify installed sorters
si.installed_sorters()

## 5.1 Combine segments to analyze

In [ ]:
# Select segments to analyze
pre_segment = si.select_segment_recording(recording_final, 0)
VNS_segment = si.select_segment_recording(recording_final, 1)
post_segment = si.select_segment_recording(recording_final, 2)

recording_to_analyze = si.ConcatenateSegmentRecording([pre_segment,VNS_segment,post_segment])

In [ ]:
# Check if file already exists to load it or save if not
if (cortex_folder / "preprocessed").is_dir():
    recording_to_analyze = si.load_extractor(cortex_folder / "preprocessed")
else:
    recording_to_analyze = recording_to_analyze.save(folder=cortex_folder / "preprocessed", **job_kwargs)

## 5.2 Run sorter locally
Here are the original parameters : {'do_correction': True, 'Th_universal': 9, 'Th_learned': 8, 'nt': 61}
Kilosort4 automatically sets the "good" units in Phy based on a <10% estimated contamination rate with spikes from other neurons (computed from the refractory period violations relative to expected).

In [ ]:
# Adapted sorting parameters in our conditions
new_nt = int(fs * 0.002+1)
params_KS4 = {'do_correction': True, 'nt' : new_nt,'duplicate_spike_ms' : 0.4,'nearest_templates': 32,'Th_universal' : 8, 'Th_learned' : 10} 

In [ ]:
KS_output_folder = cortex_folder / 'results_KS4'
print(KS_output_folder)

In [ ]:
#Running Kilosort 4
sorting_KS4 = si.run_sorter(sorter_name='kilosort4', recording=recording_to_analyze, remove_existing_folder=True,
                             folder=KS_output_folder,
                             verbose=True, **params_KS4)

In [ ]:
#Checking the output object
sorting_KS4

print(f'Spike train of a unit: {sorting_KS4.get_unit_spike_train(unit_id=1)}')
print(f'Spike train of a unit (in s): {sorting_KS4.get_unit_spike_train(unit_id=1, return_times=True)}')

We can use `widgets` functions for some quick visualizations:

In [ ]:
#Plot raster plot
%matplotlib inline
w_rs = si.plot_rasters(sorting_KS4, time_range=(0, 60*55), backend="matplotlib",figsize=(120, 5))
plt.show()

We can also save a spike sorting output for future use:

In [ ]:
# Save data 
sorting_saved_KS4 = sorting_KS4.save(folder=cortex_folder / "sorting_KS4",overwrite=True)

# 6. Extracting spiking unit metrics and visual inspection <a class="anchor" id="spiking-unit-metrics"></a>

The core of postprocessing spike sorting results revolves around extracting waveforms from paired recording-sorting objects.

In the SI API, waveforms are extracted using the `WaveformExtractor` class in the `core` module.

The `WaveformExtractor` object has convenient functions to retrieve waveforms and templates.


## 6.1 Creation of the analyzer and calculations of metrics

In [ ]:
# Load data
recording_to_analyze = si.load_extractor(cortex_folder  / "preprocessed")
sorting = si.load_extractor(cortex_folder / "sorting_KS4")

In [ ]:
# Define the folder path
analyzer_folder = cortex_folder / "spike_results"

# Check if analyzer exists
if analyzer_folder.is_dir():
    print(f"Analyzer already exists. Fast loading from {analyzer_folder}")
    analyzer = si.load_sorting_analyzer(folder=analyzer_folder)
else:
    # If it doesn't exist, create it and save it to the hard drive
    print(f"Creating the analyzer (saving to {analyzer_folder})...")
    analyzer = si.create_sorting_analyzer(
        sorting=sorting, 
        recording=recording_to_analyze,
        format="binary_folder", 
        folder=analyzer_folder)
    
print(analyzer)

In [ ]:
# Definition of how and what to calculate
n_cpus = os.cpu_count()
n_jobs = n_cpus - 4

job_kwargs = dict(n_jobs=n_jobs, chunk_duration="1s", progress_bar=True)

compute_dict = {
    'random_spikes': {'method': 'uniform', 'max_spikes_per_unit': 500},
    'waveforms': {'ms_before': 1.0, 'ms_after': 2.0},
    'templates': {'operators': ["average", "median", "std"]},
    'template_metrics' : {'include_multi_channel_metrics':True},
    'spike_amplitudes' : {},
    'template_similarity' : {},
    'correlograms' : {},
    'isi_histograms' : {'window_ms':50.0,'bin_ms':1.0,'method':"auto"},
    'unit_locations' : {'method': "monopolar_triangulation"},
    'spike_locations' : {'method' : "monopolar_triangulation",'ms_before' : 0.5,'ms_after' : 0.5},
    'noise_levels' : {},
    'quality_metrics' : {},
    'principal_components' : {'n_components': 3,'mode' : "by_channel_local"}
}

In [ ]:
# Calculate variables of the analyzer
analyzer.compute(compute_dict, **job_kwargs)
print(analyzer)

## 6.2 Visual inspection

### 6.2.1 General unit properties

In [ ]:
wf = analyzer.get_extension("waveforms")
for unit_id in analyzer.unit_ids:
    wf_data = wf.get_waveforms_one_unit(unit_id)
    spiketrain = sorting.get_unit_spike_train(unit_id)
    print(f"Unit {unit_id} - num waveforms: {wf_data.shape[0]} - num spikes: {len(spiketrain)}")

### 6.2.2 Plot of waveforms, amplitude and spikes on trace

In [ ]:
#Plot waveforms
%matplotlib widget
w = si.plot_unit_templates(analyzer,backend="ipywidgets")
plt.show()

In [ ]:
#Plot amplitude per unit
%matplotlib widget
si.plot_amplitudes(analyzer, backend="ipywidgets")

In [ ]:
# Plot spikes on trace
%matplotlib widget
si.plot_spikes_on_traces(analyzer,segment_index=0,time_range=(0, 60),backend= 'ipywidgets',order_channel_by_depth=True)
plt.show()

### 6.2.3 Plot Cross- and Autocorrelograms

In [ ]:
#Plot Correlograms
%matplotlib inline
backend_kwargs = dict(figsize=(15, 20))
#si.plot_crosscorrelograms(good_data,backend= 'matplotlib',**backend_kwargs)
si.plot_autocorrelograms(analyzer,backend= 'matplotlib',**backend_kwargs)
plt.show()

# 7. Curation <a class="anchor" id="curation"></a>

### Load data if needed

In [ ]:
analyzer = si.load_sorting_analyzer(folder=cortex_folder/"spike_results")

## 7.1 List of curation parameters : 

### 7.1.1 Defining classification rules

In [ ]:
# Function to create the correct query to curate data
def generate_query_from_lambda_rules(rules):
    """
    Generate a query string from a dictionary of lambda-based rules.
    The dictionary keys are column names, and the values are lambdas with conditions.
    """
    x = sp.Symbol('x')  # Symbolic variable for evaluation
    query_parts = []
    
    for col, condition in rules.items():
        try:
            # Evaluate the lambda for the symbolic variable
            condition_expr = condition(x)  # Example: lambda x: x > 2 becomes x > 2
            
            # Check if the result is a sympy Relational expression
            if isinstance(condition_expr, sp.core.relational.Relational):
                operator = condition_expr.rel_op  # Extract the operator (e.g., '>')
                threshold = condition_expr.rhs   # Extract the right-hand side (threshold value)
                query_parts.append(f"{col} {operator} {threshold}")
            else:
                raise ValueError(f"Condition for column '{col}' is not a valid relational expression.")
        
        except Exception as e:
            print(f"Error processing condition for column '{col}': {e}")
    
    return " | ".join(query_parts)

#### 7.1.1.a) Defining bad template metrics

In [ ]:
# num_positive_peaks, num_negative_peaks 
maxNpeaks = 2
maxNTroughs = 1

# peak_to_valley 
maxWvDuration = 1150/1000000 #in µs
minWvDuration = 100/1000000 #in µs

#Spatial decay (not available on all units)
# minSpatialDecaySlopeExp = 0.01,  # in a.u / um
# maxSpatialDecaySlopeExp = 0.1,  # in a.u / um


# Non-somatic spikes
maxPeakToTroughRatio = -0.8 # abs(peak_trough_ratio)>maxPeakToTroughRatio

In [ ]:
# Define noise rules
noise_rules = {
    #"peak_to_valley": lambda x: x < minWvDuration or x > maxWvDuration,
    "num_positive_peaks": lambda x: x > maxNpeaks,
    "num_negative_peaks": lambda x: x > maxNTroughs,
    "peak_trough_ratio": lambda x: x < maxPeakToTroughRatio
}

noise_side_rules = {
    "peak_to_valley": lambda x: x < minWvDuration or x > maxWvDuration,
}

# Creating the noise_rules query
noise_query = generate_query_from_lambda_rules(noise_rules)
print("Generated Query:", noise_query)

#### 7.1.1.b) Defining bad quality metrics

In [ ]:
maxISIviolations_ratio = 0.25 # Collaborators' value at Max 25% ISI violations, [(Lee et al., 2023)] Bomcell put 0.1
minSNR = 5 # Minimum Signal/Noise ration of 5 by bombcell, but WaveMap says 3 [(Lee et al., 2023)]
minIsoD = 20 # Minimum isolation distance of 20
maxContamPct = 10 #Max of 50% contamination pourcent (10% being what's considered acceptable but sounds a lot [Pascal Quilichinni, detection MUA]
minFiringRate = 0.1  # Firing rate at least 0.1Hz
maxAmplitudeCutoff = 0.2 # max percentage of missing spikes (Bombcell)

# Advised but not compulsory
minPresenceRatio = 0.7  # Present in at least 70% of the recording, inappropriate for our study
maxLratio = 0.3 # Maximum L-ratio of 0.3 (Bombcell)
maxFiringRate = 100  # Firing rate at most 100Hz

## Possible curation items : 
# maxLratio = 0.1 # Maximum L-ratio of 0.1, too strict for our condition
# minPresenceRatio = 0.7 #inappropriate for our study because of anesthesia + stroke
#threshold_firing_rate=(0.5, 100),  # Firing rate between 0.5 and 100 Hz

#Bombcell 
# "maxDrift": 100,  # in um --> but drift corrected so not interesting ?
# minAmplitude = 40 #in µV
# "maxPercSpikesMissing": 20,  # max percentage of missing spikes, I wonder if it is based on the gaussian fit of spike amplitude...

In [ ]:
MUA_rules = {
    "ContamPct": lambda x: x > maxContamPct,
    "isi_violations_ratio": lambda x: x > maxISIviolations_ratio,
    "snr": lambda x: x < minSNR,
    "isolation_distance": lambda x: x < minIsoD,
    "firing_rate": lambda x: x < minFiringRate,
    "amplitude_cutoff": lambda x: x > maxAmplitudeCutoff
}

MUA_side_rules = {
    "presence_ratio": lambda x: x < minPresenceRatio,
    "KSLabel": lambda x: x == "mua",
    "l_ratio": lambda x: x > maxLratio,
    "firing_rate": lambda x: x > maxFiringRate
    
}

# Creating the noise_rules query
MUA_query = generate_query_from_lambda_rules(MUA_rules)
print("Generated Query:", MUA_query)

### 7.1.2 Template metrics

Let's calcultate the following parameters : 

- **“peak_to_valley”:** duration in s between negative and positive peaks *(aka through-to-peak)*

- **“halfwidth”:** duration in s at 50% of the amplitude

- **"peak_to_trough_ratio”:** ratio between negative and positive peaks

- **“recovery_slope”:** speed to recover from the negative peak to 0

- **“repolarization_slope”:** speed to repolarize from the positive peak to 0

- **“num_positive_peaks”:** the number of positive peaks

- **“num_negative_peaks”:** the number of negative peaks

<ins>Multichannel parameters : </ins>
- **“velocity_above”:** the velocity in µm above the max channel of the template

- **“velocity_below”:** the velocity in µm below the max channel of the template

- **“exp_decay”:** the exponential decay in µm of the template amplitude over distance

- **“spread”:** the spread in µm of the template amplitude over distance

In [ ]:
#Extract template data
tm = analyzer.get_extension("template_metrics")
tm_data = tm.get_data()

In [ ]:
# Calculate violations
tm_violations = pd.DataFrame({
    col: tm_data[col].apply(rule) for col, rule in noise_rules.items()
})

# Add a new column for the count of tm_violations per row
tm_data["tm_violations_count"] = tm_violations.sum(axis=1)

# Sort the DataFrame by 'tm_violations_count' in descending order
tm_data_sorted = tm_data.sort_values(by="tm_violations_count", ascending=False)

# Function to apply styling to each cell
def highlight_tm_cells(val, column):
    if pd.isna(val):
        return 'background-color: orange'  # Highlight missing values in orange
    
    elif column in noise_rules and noise_rules[column](val):
        return 'background-color: red'

    elif column in noise_side_rules and noise_side_rules[column](val):
        return 'background-color: yellow'
    return ''

# Apply highlighting
def highlight_tm_dataframe(df):
    def highlight_tm_row(row):
        return [
            highlight_tm_cells(row[col], col) for col in df.columns
        ]
    return pd.DataFrame([highlight_tm_row(row) for _, row in df.iterrows()],
                        columns=df.columns, index=df.index)

In [ ]:
# Display the styled DataFrame
tm_data_sorted_styled = tm_data_sorted.style.apply(highlight_tm_dataframe, axis=None)
tm_data_sorted_styled

In [ ]:
# Export the dataframe as an excel file
excel_output_dir = cortex_folder/"excel_files"
os.makedirs(excel_output_dir, exist_ok=True)  # Create the directory if it doesn't exist
recording_id = mice_name + '_' + recording_name
excel_metric_filename = os.path.join(excel_output_dir, f'metrics_{cortex}_{recording_id}.xlsx')

with pd.ExcelWriter(excel_metric_filename) as writer:  # Removed quotes
    tm_data_sorted_styled.to_excel(writer, sheet_name='Template_metrics', header=True, index=True, index_label=True)

### 7.1.3 Quality metrics

The `qualitymetrics` module also provides several functions to compute qualitity metrics to validate the spike sorting results.

See also this : https://spikeinterface.readthedocs.io/en/latest/modules/qualitymetrics.html

Let's see what metrics are available:
- Amplitude cutoff (amplitude_cutoff)
- Amplitude CV (amplitude_cv_median, amplitude_cv_range)
- Amplitude median (amplitude_median)
- D-prime (d_prime)
- Drift metrics (drift_ptp, drift_std, drift_mad)
- Firing range (firing_range)
- Firing rate (firing_rate)
- Inter-spike-interval (ISI) violations (isi_violation, rp_violation)
- Isolation distance (isolation_distance)
- L-ratio (l_ratio)
- Nearest Neighbor Metrics (nn_hit_rate, nn_miss_rate, nn_isolation, nn_noise_overlap)
- Noise cutoff (not currently implemented)
- Presence ratio (presence_ratio)
- Standard Deviation (SD) ratio (sd_ratio)
- Silhouette score (silhouette, silhouette_full)
- Sliding refractory period violations (sliding_rp_violations)
- Signal-to-noise ratio (snr)
- Synchrony Metrics (synchrony)

In [ ]:
#Extract quality metrics
r_filenameTSV = cortex_folder/"results_KS4/sorter_output/cluster_ContamPct.tsv"
contam_pct = pd.read_csv(r_filenameTSV, sep='\t')

r_filenameTSV = cortex_folder/"results_KS4/sorter_output/cluster_KSLabel.tsv"
KSLabel = pd.read_csv(r_filenameTSV, sep='\t')

r_filenameTSV = cortex_folder/"results_KS4/sorter_output/cluster_group.tsv"
cluster_group = pd.read_csv(r_filenameTSV, sep='\t')

In [ ]:
qm = analyzer.get_extension("quality_metrics")
qm_data = qm.get_data()

col_to_display = ['KSLabel','ContamPct', 'num_spikes','firing_rate','presence_ratio','snr','amplitude_cutoff','isi_violations_ratio','isolation_distance','l_ratio','qm_violations_count']

qm_data_label = pd.concat([KSLabel['KSLabel'],qm_data, contam_pct['ContamPct']], axis=1)

In [ ]:
# Calculate violations
qm_violations = pd.DataFrame({
    col: qm_data_label[col].apply(rule) for col, rule in MUA_rules.items()
})

# Add a new column for the count of qm_violations per row
qm_violations_count = qm_violations.sum(axis=1)


qm_data_label["qm_violations_count"] = qm_violations_count

# Sort the DataFrame by 'qm_violations_count' in descending order
qm_data_sorted = qm_data_label.sort_values(
    by=['KSLabel', 'qm_violations_count', 'num_spikes'], 
    ascending=[True, False, False]
)

# Function to apply styling to each cell
def highlight_qm_cells(val, column):
    # Check if the value is NA
    if pd.isna(val):
        return 'background-color: orange'  # Highlight missing values in orange

    # Check against the main rules
    elif column in MUA_rules and MUA_rules[column](val):
        return 'background-color: red'
    
    # Check against the side rules
    elif column in MUA_side_rules and MUA_side_rules[column](val):
        return 'background-color: yellow'
    
    # Default style (no highlight)
    return ''

# Apply highlighting
def highlight_qm_dataframe(df):
    def highlight_qm_row(row):
        return [
            highlight_qm_cells(row[col], col) for col in df.columns
        ]
    return pd.DataFrame([highlight_qm_row(row) for _, row in df.iterrows()],
                        columns=df.columns, index=df.index)

In [ ]:
# Display table of quality metrics
qm_data_sorted_styled = qm_data_sorted[col_to_display].style.apply(highlight_qm_dataframe, axis=None)
qm_data_sorted_styled

In [ ]:
#Export data to Excel
with pd.ExcelWriter(excel_metric_filename,
                    mode='a') as writer:  # Removed quotes
    qm_data_sorted_styled.to_excel(writer, sheet_name='Quality_metrics', header=True, index=True, index_label=True)

## 7.2 Metric-based and manual curation

### 7.2.1 Defining Noise / MUA / Good units

In [ ]:
# We assume 'noise_query' selects units with Bad Template Metrics
# We assume 'MUA_query' selects units with Bad Quality Metrics
bad_template_ids = tm_data.query(noise_query).index.values
bad_quality_ids = qm_data_sorted.query(MUA_query).index.values

# NOISE = Bad Template AND Bad Quality (Intersection)
noise_unit_ids = np.intersect1d(bad_template_ids, bad_quality_ids)

# MUA = (Bad Template OR Bad Quality) MINUS Noise
# First, find everyone who failed at least one test (Union)
any_bad_ids = np.union1d(bad_template_ids, bad_quality_ids)
# Then remove the ones we already marked as Noise
MUA_unit_ids = np.setdiff1d(any_bad_ids, noise_unit_ids)

# GOOD = The Remainder (Everyone else)
# We start with ALL units and remove the MUA/Noise ones
all_unit_ids = analyzer.unit_ids
excluded_ids = np.union1d(noise_unit_ids, MUA_unit_ids)
good_unit_ids = np.setdiff1d(all_unit_ids, excluded_ids)

In [ ]:
print(f"Classification Results:")
print(f"  - Good:  {len(good_unit_ids)}")
print(f"  - MUA:   {len(MUA_unit_ids)}")
print(f"  - Noise: {len(noise_unit_ids)}")

### 7.2.2 Apply labels and extra quality metrics to the sorting object

In [ ]:
# Apply 'quality_label' so the GUI can color-code them.
sorting = analyzer.sorting

# Reset everything to 'mua' first (Default bucket)
sorting.set_property(key='quality_label', values=['mua'] * sorting.get_num_units())

# Label Noise (displayed as Black/Gray)
if len(noise_unit_ids) > 0:
    sorting.set_property(key='quality_label', values=['noise'] * len(noise_unit_ids), ids=noise_unit_ids)

# Label Good (displayed as Green)
if len(good_unit_ids) > 0:
    sorting.set_property(key='quality_label', values=['good'] * len(good_unit_ids), ids=good_unit_ids)

In [ ]:
# Adding extra quality metrics
sorting.set_property(key='qm_violations_count', values=qm_violations_count)

### 7.2.3 Manual curation with SpikeInterface GUI

In [ ]:
# This prevents it from searching for the missing PySide6
os.environ["QT_API"] = "pyqt5"

try:
    %gui qt
except:
    print("Could not set %gui qt. If window freezes, restart kernel.")

print("Launching SpikeInterface GUI...")
# curation=True enables the 'Merge' and 'Trash' buttons
wi.plot_sorting_summary(analyzer, curation=True, backend="spikeinterface_gui")

## 7.3 Recompute metrics & export after manual curation

### 7.3.1 Load and apply the manual curation to the original sorting

In [ ]:
curation_file = cortex_folder / "spike_results/spikeinterface_gui/curation_data.json"

if not curation_file.exists():
    print("No curation.json found! Did you click 'Save' in the GUI?")
    # Keeping the original analyzer if nothing is found
    sorting_curated = sorting
else:
    print("Curation file found! Applying manual curation (merges/labels)...")
    #Open curation file if existing
    with open(curation_file, 'r') as f:
        curation_dict = json.load(f)
        
    #Apply curation
    sorting_curated = si.apply_curation(sorting, curation_dict)
    
print(f"Original number of units: {sorting.get_num_units()}")
print(f"Curated number of units: {sorting_curated.get_num_units()}")

### 7.3.2 Create and compute a new analyzer for the curated data

In [ ]:
analyzer_curated_folder = cortex_folder / "spike_results_curated"
if analyzer_curated_folder.exists():
    shutil.rmtree(analyzer_curated_folder) # Clean up previous runs

if (cortex_folder / "preprocessed").is_dir():
    recording_to_analyze = si.load_extractor(cortex_folder / "preprocessed")
else:
    print("Verify you already have an existing 'preprocessed' folder, or run the preprocessing step first.")

analyzer_curated = si.create_sorting_analyzer(
    sorting=sorting_curated, 
    recording=recording_to_analyze, 
    format="binary_folder", 
    folder=analyzer_curated_folder
)

In [ ]:
# Definition of how and what to calculate
n_cpus = os.cpu_count()
n_jobs = n_cpus - 4

job_kwargs = dict(n_jobs=n_jobs, chunk_duration="1s", progress_bar=True)

compute_dict_light = {
    'random_spikes': {'method': 'uniform', 'max_spikes_per_unit': 500},
    'waveforms': {'ms_before': 1.0, 'ms_after': 2.0},
    'templates': {'operators': ["average", "median", "std"]},
    'template_metrics' : {'include_multi_channel_metrics':True},
    'spike_amplitudes' : {},
    'correlograms' : {},
    'isi_histograms' : {'window_ms':50.0,'bin_ms':1.0,'method':"auto"},
    'unit_locations' : {'method': "monopolar_triangulation"},
    'spike_locations' : {'method' : "monopolar_triangulation",'ms_before' : 0.5,'ms_after' : 0.5},
    'noise_levels' : {},
    'quality_metrics' : {}
}

In [ ]:
# Recompute metrics for the new merged/split units
print("Recomputing metrics for curated units...")
analyzer_curated.compute(compute_dict_light, **job_kwargs)

### 7.3.5 Extract labels added in the GUI

In [ ]:
final_unit_ids = sorting_curated.unit_ids

# Extract manually curated and automatic labels
manual_labels = sorting_curated.get_property('quality')
auto_labels = sorting_curated.get_property('quality_label')

final_labels = []

# Loop on units to check if they have been manually reassigned
for i in range(len(final_unit_ids)):
    m_label = manual_labels[i]
    a_label = auto_labels[i]

    # If the manual label is empty, fallback to the automatic pre-sorting
    if m_label == '':
        final_labels.append(a_label)
    else:
        # If filled, the manual human decision overrides!
        final_labels.append(m_label)

# Convert our clean list into a Numpy array for boolean filtering
final_labels = np.array(final_labels)

# Final sorting masks
good_ids = final_unit_ids[final_labels == 'good']
mua_ids = final_unit_ids[final_labels == 'mua']
# For noise, we catch both 'noise' and 'artifact' tags just in case
noise_ids = final_unit_ids[(final_labels == 'noise') | (final_labels == 'artifact')]

# Summary print to double-check the math
print(f" - Good units: {len(good_ids)}")
print(f" - MUA: {len(mua_ids)}")
print(f" - Noise: {len(noise_ids)}")
print(f" - Total processed: {len(good_ids) + len(mua_ids) + len(noise_ids)} / {len(final_unit_ids)}")

### 7.3.6 Export metrics to Excel based on GUI labels

In [ ]:
#Getting updated metrics
qm_data_curated = analyzer_curated.get_extension('quality_metrics').get_data().copy()
tm_data_curated = analyzer_curated.get_extension('template_metrics').get_data().copy()

#Gathering extra metrics
contam_pct = sorting_curated.get_property('ContamPct')
qm_violations = sorting_curated.get_property('qm_violations_count')

base_labels = {
    'quality': manual_labels,
    'quality_label': auto_labels,
    'final_label': final_labels
}
df_labels_only = pd.DataFrame(base_labels, index=final_unit_ids)

qm_front_columns = base_labels.copy()

if qm_violations is not None:
    qm_front_columns['qm_violations_count'] = qm_violations

if contam_pct is not None:
    qm_front_columns['ContamPct'] = contam_pct

df_front_qm = pd.DataFrame(qm_front_columns, index=final_unit_ids)

# Fusing dataframes
qm_master = df_front_qm.join(qm_data_curated)
tm_master = df_labels_only.join(tm_data_curated)

# 4. Color Rules & Export (The Robust Column-by-Column Method)
columns_to_ignore_for_colors = ['quality', 'quality_label']

# We define a function that processes ONE column at a time
def apply_qm_colors(col_series):
    col_name = col_series.name # Get the exact name of the current column
    
    # If it's a summary/text column, we return an empty style for the whole column
    if col_name in columns_to_ignore_for_colors:
        return [''] * len(col_series)
    
    # Otherwise, we apply exact rules to every value in this column
    return [highlight_qm_cells(val, col_name) for val in col_series]

def apply_tm_colors(col_series):
    col_name = col_series.name
    
    if col_name in columns_to_ignore_for_colors:
        return [''] * len(col_series)
        
    return [highlight_tm_cells(val, col_name) for val in col_series]

# Export
excel_output_dir = cortex_folder / "excel_files"
os.makedirs(excel_output_dir, exist_ok=True)
excel_metric_filename = excel_output_dir / f'metrics_{cortex}_curated_{recording_id}.xlsx'

with pd.ExcelWriter(excel_metric_filename, engine='openpyxl') as writer:
    qm_master.style.apply(apply_qm_colors, axis=0) \
             .to_excel(writer, sheet_name='All_Quality_Metrics')
             
    tm_master.style.apply(apply_tm_colors, axis=0) \
             .to_excel(writer, sheet_name='All_Template_Metrics')

print(f"Export successful in: {excel_metric_filename}")

# 8. Data inspection and pre-classification INT/PYR <a class="anchor" id="INT_PYR_classification"></a>

## 8.1 Import analyzer and isolate good units

In [ ]:
# Define the path (in case it wasn't defined earlier in this cell)
analyzer_curated_folder = cortex_folder / "spike_results_curated"

# Load curated analyzer data
if analyzer_curated_folder.is_dir():
    print(f"Loading of the analyzer from {analyzer_curated_folder}...")
    analyzer_curated = si.load_sorting_analyzer(folder=analyzer_curated_folder)
else:
    print("Folder not found. Make sure the creation and curation steps were properly executed!")

print(analyzer_curated)

In [ ]:
#Create a "Good units only" analyzer (kept in memory for speed)
good_data = analyzer_curated.select_units(
    unit_ids=good_ids, 
    format="memory"
)

#Define folder for good curated data curated_image_output_dir
image_output_dir = cortex_folder/"images_files"
recording_id = mice_name + '_' + recording_name
os.makedirs(image_output_dir, exist_ok=True)  # Create the directory if it doesn't exist

## 8.2 Waveform visualization

In [ ]:
%matplotlib widget
good_w = si.plot_unit_templates(good_data,backend="ipywidgets")
plt.show()

## 8.3 Raster plot (without or with traces on the back)

In [ ]:
%matplotlib inline
good_units_ids = good_data.unit_ids
good_w_rs = si.plot_rasters(sorting, time_range=(0, 60*60), backend="matplotlib",unit_ids=good_units_ids,figsize=(50, 10))


#Save the figure
fig_name = 'good_curated_rasterplot'
filename = os.path.join(image_output_dir, f'{fig_name}_{cortex}_{recording_id}.png')
plt.savefig(filename)

#Display the figure
plt.show()

In [ ]:
%matplotlib widget
#backend_kwargs = dict(figsize=(15, 10))
si.plot_spikes_on_traces(good_data,segment_index=0,time_range=(0, 60),backend= 'ipywidgets',order_channel_by_depth=True)#,**backend_kwargs)
plt.show()

## 8.4 Unit classification based on template metrics

In [ ]:
# Extract template metrics data
good_tm = good_data.get_extension("template_metrics")
good_tm_data = good_tm.get_data()

In [ ]:
# 2D plot
%matplotlib widget
si.plot_template_metrics(good_data, include_metrics=["peak_to_valley", "half_width","repolarization_slope","peak_trough_ratio"], 
                         backend="ipywidgets")
plt.show()

In [ ]:
# 3D plot
# Use the row index as the unit ID
good_tm_data["unit_id"] = good_tm_data.index.astype(str)

# Assign a unique color to each unit ID (row)
unique_units = good_tm_data["unit_id"]
colors = cm.tab20(range(len(unique_units)))  # Generate distinct colors
color_map = dict(zip(unique_units, colors))  # Map unit IDs to colors

# Function to update the plot
def update_plot(x_axis, y_axis, z_axis):
    fig = plt.figure(figsize=(8, 6))
    ax = fig.add_subplot(111, projection='3d')

    # Plot each unit with its assigned color
    for unit_id, color in color_map.items():
        unit_data = good_tm_data[good_tm_data["unit_id"] == unit_id]
        ax.scatter(
            unit_data[x_axis],
            unit_data[y_axis],
            unit_data[z_axis],
            label=f"Unit {unit_id}",  # Add unit ID to legend
            color=color,
            s=50
        )
    
    # Set labels and title
    ax.set_xlabel(x_axis)
    ax.set_ylabel(y_axis)
    ax.set_zlabel(z_axis)
    ax.set_title(f"3D Plot: {x_axis} vs {y_axis} vs {z_axis}")
    
    # Add a legend
    ax.legend(title="Unit IDs", bbox_to_anchor=(1.05, 1), loc="upper left")
    
    plt.show()

# Dropdowns for selecting axes
x_axis_dropdown = widgets.Dropdown(
    options=good_tm_data.columns[:-1],  # Exclude "unit_id" from the options
    value=good_tm_data.columns[0],  # Default to the first column
    description="X-Axis:"
)
y_axis_dropdown = widgets.Dropdown(
    options=good_tm_data.columns[:-1],
    value=good_tm_data.columns[1],  # Default to the second column
    description="Y-Axis:"
)
z_axis_dropdown = widgets.Dropdown(
    options=good_tm_data.columns[:-1],
    value=good_tm_data.columns[2],  # Default to the third column
    description="Z-Axis:"
)

# Create interactive widget
%matplotlib widget
interactive_plot = widgets.interactive(
    update_plot,
    x_axis=x_axis_dropdown,
    y_axis=y_axis_dropdown,
    z_axis=z_axis_dropdown
)

# Display the interactive plot with dropdowns
display(interactive_plot)

## 8.5 Autocorrelograms 

In [ ]:
%matplotlib inline

backend_kwargs = dict(figsize=(15, 20))
#si.plot_crosscorrelograms(good_data,backend= 'matplotlib',**backend_kwargs)
si.plot_autocorrelograms(good_data,backend= 'matplotlib',**backend_kwargs)

#Save the figure
fig_name = 'good_curated_autocorrelograms'
filename = os.path.join(image_output_dir, f'{fig_name}_{cortex}_{recording_id}.png')
plt.savefig(filename)

plt.show()

## 8.6 Display unit and spike locations

When using silicon probes, we can estimate the unit (or spike) location with triangulation. This can be done either with a simple center of mass or by assuming a monopolar model:

$$V_{ext}(\boldsymbol{r_{ext}}) = \frac{I_n}{4 \pi \sigma |\boldsymbol{r_{ext}} - \boldsymbol{r_{n}}|}$$

where $\boldsymbol{r_{n}}$ is the position of the neuron, and $\boldsymbol{r_{n}}$ of the electrode(s).

In [ ]:
%matplotlib inline

backend_kwargs = dict(figsize=(20, 20), plot_legend=True)

#Spike locations
good_spike_locations = si.plot_spike_locations(good_data,backend= 'matplotlib',**backend_kwargs)
good_spike_locations.ax.set_ylim(-100, 400)

fig_name = 'good_curated_spike_locations'
filename = os.path.join(image_output_dir, f'{fig_name}_{cortex}_{recording_id}.png')
plt.savefig(filename)


# Unit locations
good_unit_locations = si.plot_unit_locations(good_data,backend= 'matplotlib',**backend_kwargs)

fig_name = 'good_curated_unit_locations'
filename = os.path.join(image_output_dir, f'{fig_name}_{cortex}_{recording_id}.png')
plt.savefig(filename)

plt.show()

## 8.7 Adjust timing of the original recording
Adjust timing of spikes to add "silence" instead of interferences

In [ ]:
# load spike_times and removed timepoint_array
spike_times_folder = cortex_folder / 'results_KS4/sorter_output/'
spike_times = np.load(spike_times_folder / 'spike_times.npy')
timepoint_array = np.load(cortex_folder / 'timepoint_array.npy')

# Create a sorted list of removal endpoints
removal_points = np.concatenate([np.arange(start, end) for start, end in timepoint_array])

# Sort the removal points (already sorted in this case, but ensuring robustness)
removal_points.sort()

# Use searchsorted to determine how many points were removed before each spike
correction = np.searchsorted(removal_points, spike_times, side='right')

# Adjust spike times
adjusted_spike_times = spike_times + correction

np.save(spike_times_folder / 'adjusted_spike_times.npy', adjusted_spike_times)

# Save good clusters' indices
np.save(spike_times_folder / 'good_unit_ids.npy', good_unit_ids)

## 8.8 Population Dynamics (Neural Trajectories) with Real-Time Alignment
To understand how the entire network behaves over time, we compute the Principal Component Analysis (PCA) on the smoothed population firing rates. 
Because VNS artifacts were removed during preprocessing, we first realign the spike times to absolute experimental time using a `searchsorted` correction before binning the data.

* **Pre-VNS:** 0 to 15 min
* **VNS:** 15 to 45 min
* **Post-VNS:** 45 to 60 min

### 8.7.1 Global Parameters for Time Bins

In [ ]:
bin_size = 1  # 1 second bins for a 1-hour recording
total_duration = 60 * 60  # 60 minutes in seconds
fs = analyzer_curated.sampling_frequency

pre_end = 15 * 60       # 15 minutes
vns_end = 45 * 60       # 45 minutes

# Create time bins and logical masks for plotting
bins = np.arange(0, total_duration + bin_size, bin_size)
time_centers = bins[:-1] + bin_size / 2

pre_mask = time_centers < pre_end
vns_mask = (time_centers >= pre_end) & (time_centers < vns_end)
post_mask = time_centers >= vns_end

### 8.7.2 Define the PCA Plotting Function

In [ ]:
def plot_population_pca(unit_ids_to_process, title_suffix, filename_suffix):
    num_neurons = len(unit_ids_to_process)
    if num_neurons == 0:
        print(f"⚠️ No units found for {title_suffix}. Skipping plot.")
        return
        
    print(f"Processing {num_neurons} units for: {title_suffix}...")
    spike_counts = np.zeros((num_neurons, len(time_centers)))

    for i, unit_id in enumerate(unit_ids_to_process):
        # 1. Get raw indices and adjust to real absolute time
        spike_indices = sorting_curated.get_unit_spike_train(unit_id, return_times=False)
        correction = np.searchsorted(removal_points, spike_indices, side='right')
        adjusted_times_sec = (spike_indices + correction) / fs
        
        # 2. Count spikes in bins
        counts, _ = np.histogram(adjusted_times_sec, bins=bins)
        spike_counts[i, :] = counts

    # 3. Smooth the firing rates (5-second window)
    smoothed_rates = gaussian_filter1d(spike_counts, sigma=5, axis=1)
    X = smoothed_rates.T  # Shape becomes (Time x Neurons)

    # 4. Standardize (Z-score) to balance MUA and Good units
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # 5. Compute PCA
    pca_spiketimes = PCA(n_components=2)
    pca_spiketimes_result = pca_spiketimes.fit_transform(X_scaled)
    explained_var = pca_spiketimes.explained_variance_ratio_ * 100

    # 6. Plot the Trajectories
    %matplotlib inline
    fig, ax = plt.subplots(figsize=(12, 10))
    
    ax.scatter(pca_spiketimes_result[pre_mask, 0], pca_spiketimes_result[pre_mask, 1], 
               color='royalblue', s=15, alpha=0.6, label='Pre-VNS (0-15 min)')
    ax.scatter(pca_spiketimes_result[vns_mask, 0], pca_spiketimes_result[vns_mask, 1], 
               color='crimson', s=15, alpha=0.6, label='VNS (15-45 min)')
    ax.scatter(pca_spiketimes_result[post_mask, 0], pca_spiketimes_result[post_mask, 1], 
               color='forestgreen', s=15, alpha=0.6, label='Post-VNS (45-60 min)')

    # Aesthetics
    ax.set_title(f"Neural Population Trajectories ({title_suffix}) - {cortex.capitalize()} Cortex", fontsize=16, fontweight='bold')
    ax.set_xlabel(f"Principal Component 1 ({explained_var[0]:.1f}% variance)", fontsize=12)
    ax.set_ylabel(f"Principal Component 2 ({explained_var[1]:.1f}% variance)", fontsize=12)
    ax.legend(loc='best', fontsize=12, markerscale=2)
    ax.grid(True, linestyle='--', alpha=0.5)

    # Save and display
    filename = os.path.join(image_output_dir, f'population_pca_spiketimes_{filename_suffix}_{cortex}_{recording_id}.png')
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.show()

### 8.7.3 Generate and save both figures

In [ ]:
# Figure 1: Good units only
plot_population_pca(good_ids, title_suffix="Good Units Only", filename_suffix="good_only")

# Figure 2: Good units + MUA combined
all_valid_ids = np.concatenate([good_ids, mua_ids])
plot_population_pca(all_valid_ids, title_suffix="Good Units + MUA", filename_suffix="good_and_mua")